# Food Recognition Demo Inference

This notebook demonstrates the final user-facing prediction workflow. It loads the **ResNet50 FT-V2** champion, applies calibrated confidence, returns top-k predictions, and assigns a product action: **auto-accept**, **suggest**, **confirm**, or **review**.


## 1. Demo Goal

The previous notebooks produced a trained model, calibrated confidence, and a decision policy. This notebook shows how those pieces work together for one image at a time.

Expected output for each image:

1. top-k food predictions;
2. calibrated confidence;
3. top-1/top-2 margin;
4. decision band;
5. recommended user-facing action.


In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from PIL import Image
from torch import nn
from torchvision import models, transforms


In [ ]:
@dataclass(frozen=True)
class CFG:
    """Configuration for final demo inference."""

    IMAGE_SIZE: tuple[int, int] = (224, 224)
    NUM_CLASSES: int = 101
    TOP_K: int = 5
    DATA_DIR: Path = Path("/kaggle/input/datasets/kmader/food41")
    MODEL_ARTIFACT_DIR: Path = Path(
        "/kaggle/input/models/tuannm3823/food101-resnet50-refinements/"
        "pytorch/default/1"
    )
    CALIBRATION_OUTPUT_DIR: Path = Path(
        "/kaggle/input/notebooks/tuannm3823/"
        "resnet50-ft-v2-error-analysis-calibration/"
        "results/resnet50_error_calibration"
    )
    DECISION_OUTPUT_DIR: Path = Path(
        "/kaggle/input/notebooks/tuannm3823/"
        "05-confidence-decision-layer/"
        "results/confidence_decision_layer"
    )
    RESULTS_DIR: Path = Path("/kaggle/working/results/food_recognition_demo")
    CHECKPOINT_NAME: str = "resnet50_ft_v2_best.pth"
    CALIBRATION_FILE: str = "calibration_summary.csv"
    HARD_CLASS_FILE: str = "hard_classes_calibrated.csv"
    CONFUSION_FILE: str = "top_confusion_pairs_calibrated.csv"
    POLICY_FILE: str = "decision_policy.csv"
    DEMO_PREDICTIONS_FILE: str = "demo_predictions.csv"
    DEMO_SUMMARY_FILE: str = "demo_decision_summary.csv"
    DEFAULT_TEMPERATURE: float = 0.958111
    DEFAULT_AUTO_CONFIDENCE: float = 0.70
    DEFAULT_SUGGEST_CONFIDENCE: float = 0.35
    DEFAULT_MARGIN_THRESHOLD: float = 0.40
    SAMPLE_IMAGE_PATHS: tuple[str, ...] = ()


CFG.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Output directory: {CFG.RESULTS_DIR}")


## 2. Resolve Inputs

The demo can run from linked Kaggle Notebook outputs. If those outputs are not attached, it falls back to the default calibration and decision thresholds recorded in the project docs.


In [ ]:
def decision_artifact_roots() -> list[Path]:
    """Return likely roots for saved Notebook 5 decision-layer artifacts."""
    roots = [CFG.DECISION_OUTPUT_DIR]

    notebook_root = Path("/kaggle/input/notebooks")
    if notebook_root.exists():
        roots.extend(sorted(notebook_root.glob("**/results/confidence_decision_layer")))

    return list(dict.fromkeys(roots))


def resolve_image_dir(data_dir: Path) -> Path:
    """Resolve Food-101 image directory."""
    candidate_dirs = [data_dir / "images", data_dir]
    for candidate_dir in candidate_dirs:
        if not candidate_dir.exists():
            continue
        class_dirs = [path for path in candidate_dir.iterdir() if path.is_dir()]
        has_images = any(
            image_path.suffix.lower() in {".jpg", ".jpeg", ".png"}
            for class_dir in class_dirs
            for image_path in class_dir.iterdir()
        )
        if has_images:
            return candidate_dir
    raise FileNotFoundError(f"Food-101 images were not found under {data_dir}.")


def resolve_file(filename: str, roots: list[Path]) -> Path | None:
    """Resolve a file from known roots or recursive Kaggle input search."""
    candidates = []
    for root in roots:
        candidates.append(root / filename)
        if root.exists():
            candidates.extend(sorted(root.rglob(filename)))

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(sorted(kaggle_input.rglob(filename)))

    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


IMAGE_DIR = resolve_image_dir(CFG.DATA_DIR)
class_names = sorted(path.name for path in IMAGE_DIR.iterdir() if path.is_dir())
class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}

checkpoint_path = resolve_file(CFG.CHECKPOINT_NAME, [CFG.MODEL_ARTIFACT_DIR])
if checkpoint_path is None:
    raise FileNotFoundError(f"Could not find {CFG.CHECKPOINT_NAME} under Kaggle inputs.")

calibration_path = resolve_file(CFG.CALIBRATION_FILE, [CFG.CALIBRATION_OUTPUT_DIR])
hard_class_path = resolve_file(CFG.HARD_CLASS_FILE, [CFG.CALIBRATION_OUTPUT_DIR])
confusion_path = resolve_file(CFG.CONFUSION_FILE, [CFG.CALIBRATION_OUTPUT_DIR])
policy_path = resolve_file(CFG.POLICY_FILE, decision_artifact_roots())

print(f"Image directory: {IMAGE_DIR}")
print(f"Checkpoint: {checkpoint_path}")
print(f"Calibration file: {calibration_path}")
print(f"Hard-class file: {hard_class_path}")
print(f"Confusion file: {confusion_path}")
print(f"Policy file: {policy_path}")


In [ ]:
def load_temperature(path: Path | None) -> float:
    """Load calibrated temperature or fall back to project default."""
    if path is None:
        return CFG.DEFAULT_TEMPERATURE
    calibration_df = pd.read_csv(path)
    if "temperature" not in calibration_df.columns:
        return CFG.DEFAULT_TEMPERATURE
    return float(calibration_df["temperature"].iloc[0])


def load_policy(path: Path | None) -> dict[str, float]:
    """Load decision thresholds or fall back to project defaults."""
    if path is None:
        return {
            "auto_confidence": CFG.DEFAULT_AUTO_CONFIDENCE,
            "suggest_confidence": CFG.DEFAULT_SUGGEST_CONFIDENCE,
            "margin_threshold": CFG.DEFAULT_MARGIN_THRESHOLD,
        }
    policy_df = pd.read_csv(path)
    return {
        "auto_confidence": float(policy_df["auto_confidence"].iloc[0]),
        "suggest_confidence": float(policy_df["suggest_confidence"].iloc[0]),
        "margin_threshold": float(policy_df["margin_threshold"].iloc[0]),
    }


def load_hard_classes(path: Path | None) -> set[str]:
    """Load hard class names if available."""
    if path is None:
        return {"chocolate_mousse", "steak", "pork_chop", "bread_pudding", "tuna_tartare"}
    hard_df = pd.read_csv(path)
    return set(hard_df["class_name"].tolist())


def load_confusion_pairs(path: Path | None) -> set[tuple[str, str]]:
    """Load repeated confusion pairs if available."""
    if path is None:
        return set()
    confusion_df = pd.read_csv(path)
    return set(zip(confusion_df["actual"], confusion_df["predicted"]))


TEMPERATURE = load_temperature(calibration_path)
POLICY = load_policy(policy_path)
HARD_CLASSES = load_hard_classes(hard_class_path)
CONFUSION_PAIRS = load_confusion_pairs(confusion_path)

print(f"Temperature: {TEMPERATURE:.6f}")
print(f"Policy: {POLICY}")
print(f"Hard classes loaded: {len(HARD_CLASSES)}")
print(f"Confusion pairs loaded: {len(CONFUSION_PAIRS)}")


## 3. Load Champion Model

The architecture matches the ResNet50 FT-V2 checkpoint from Notebook 2.


In [ ]:
def make_classifier_head(in_features: int) -> nn.Sequential:
    """Create the project-standard Food-101 classifier head."""
    return nn.Sequential(
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, CFG.NUM_CLASSES),
    )


def build_resnet50() -> nn.Module:
    """Build ResNet50 with the project classifier head."""
    model = models.resnet50(weights=None)
    model.fc = make_classifier_head(model.fc.in_features)
    return model


model = build_resnet50()
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model = model.to(device)
model.eval()
print("Champion model loaded.")


## 4. Prediction And Action Helpers

The action logic uses calibrated confidence and top-1/top-2 margin. If the true label is known for a demo image, repeated confusion pairs can also trigger the **review** band. In real user uploads, the true label is usually unknown, so the workflow relies on confidence, margin, and hard predicted classes.


In [ ]:
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

INFERENCE_TRANSFORMS = transforms.Compose(
    [
        transforms.Resize(CFG.IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(NORM_MEAN, NORM_STD),
    ]
)


def predict_top_k(image_path: str | Path, top_k: int = CFG.TOP_K) -> pd.DataFrame:
    """Return calibrated top-k predictions for one image."""
    image_path = Path(image_path)
    image = Image.open(image_path).convert("RGB")
    image_tensor = INFERENCE_TRANSFORMS(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image_tensor).cpu()
        probabilities = F.softmax(logits / TEMPERATURE, dim=1)
        top_probabilities, top_indices = probabilities.topk(top_k, dim=1)

    return pd.DataFrame(
        [
            {
                "rank": rank + 1,
                "class_name": class_names[class_idx],
                "confidence": confidence,
            }
            for rank, (class_idx, confidence) in enumerate(
                zip(top_indices[0].tolist(), top_probabilities[0].tolist())
            )
        ]
    )


def recommend_action(predictions_df: pd.DataFrame, actual_label: str | None = None) -> dict[str, str | float]:
    """Recommend a product action from calibrated top-k predictions."""
    top_1 = predictions_df.iloc[0]
    top_2_confidence = float(predictions_df.iloc[1]["confidence"]) if len(predictions_df) > 1 else 0.0
    top_1_confidence = float(top_1["confidence"])
    predicted_label = str(top_1["class_name"])
    margin = top_1_confidence - top_2_confidence

    if actual_label is not None and (actual_label, predicted_label) in CONFUSION_PAIRS:
        decision_band = "review"
        action = "Flag for review because this matches a known confusion pair."
    elif predicted_label in HARD_CLASSES and top_1_confidence < POLICY["auto_confidence"]:
        decision_band = "confirm"
        action = "Ask the user to confirm because this is a hard predicted class."
    elif (
        top_1_confidence >= POLICY["auto_confidence"]
        and margin >= POLICY["margin_threshold"]
        and predicted_label not in HARD_CLASSES
    ):
        decision_band = "auto_accept"
        action = "Accept the top prediction automatically."
    elif top_1_confidence >= POLICY["suggest_confidence"]:
        decision_band = "suggest"
        action = "Show ranked suggestions for user selection."
    else:
        decision_band = "confirm"
        action = "Ask the user to confirm before applying a label."

    return {
        "predicted_label": predicted_label,
        "top_1_confidence": top_1_confidence,
        "top_1_top_2_margin": margin,
        "decision_band": decision_band,
        "recommended_action": action,
    }


def infer_food(image_path: str | Path, actual_label: str | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Run final demo inference for one image."""
    predictions = predict_top_k(image_path)
    action = recommend_action(predictions, actual_label=actual_label)
    return predictions, pd.DataFrame([action])


## 5. Demo Examples

Set `CFG.SAMPLE_IMAGE_PATHS` for your own images. If no paths are provided, the notebook first tries to sample examples from Notebook 5 decision-band outputs so the demo can cover **auto-accept**, **suggest**, **confirm**, and **review** cases. If those files are not attached, it falls back to deterministic Food-101 examples.

The demo exports two lightweight CSV files: `demo_predictions.csv` for top-k predictions and `demo_decision_summary.csv` for one-row-per-image decision actions.


In [ ]:
DECISION_EXAMPLE_FILES = {
    "auto_accept": "decision_examples_auto_accept.csv",
    "suggest": "decision_examples_suggest.csv",
    "confirm": "decision_examples_confirm.csv",
    "review": "decision_examples_review.csv",
}


def image_exists(path_value: str | Path) -> bool:
    """Return whether a candidate image path exists locally."""
    return Path(path_value).exists()


def decision_example_demo_images(max_per_band: int = 1) -> list[Path]:
    """Load representative image paths from Notebook 5 decision-band examples."""
    image_paths = []
    seen_paths = set()
    roots = decision_artifact_roots()

    for filename in DECISION_EXAMPLE_FILES.values():
        example_path = resolve_file(filename, roots)
        if example_path is None:
            continue
        examples_df = pd.read_csv(example_path)
        if "path" not in examples_df.columns:
            continue

        selected_count = 0
        for path_value in examples_df["path"].dropna().tolist():
            image_path = Path(path_value)
            if str(image_path) in seen_paths or not image_exists(image_path):
                continue
            image_paths.append(image_path)
            seen_paths.add(str(image_path))
            selected_count += 1
            if selected_count >= max_per_band:
                break

    return image_paths


def default_demo_images(image_dir: Path, max_examples: int = 6) -> list[Path]:
    """Return deterministic demo image paths from different classes."""
    preferred_classes = [
        "miso_soup",
        "ice_cream",
        "steak",
        "tuna_tartare",
        "chocolate_mousse",
        "greek_salad",
    ]
    image_paths = []
    for class_name in preferred_classes:
        class_dir = image_dir / class_name
        if class_dir.exists():
            candidates = sorted(
                path for path in class_dir.iterdir() if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
            )
            if candidates:
                image_paths.append(candidates[0])
        if len(image_paths) >= max_examples:
            break
    return image_paths


demo_paths = [Path(path) for path in CFG.SAMPLE_IMAGE_PATHS]
if not demo_paths:
    demo_paths = decision_example_demo_images()
if not demo_paths:
    demo_paths = default_demo_images(IMAGE_DIR)

summary_rows = []
prediction_rows = []
for image_path in demo_paths:
    actual_label = image_path.parent.name if image_path.parent.name in class_to_idx else None
    predictions, action = infer_food(image_path, actual_label=actual_label)
    summary_row = action.iloc[0].to_dict()
    summary_row["actual_label"] = actual_label
    summary_row["image_path"] = str(image_path)
    summary_rows.append(summary_row)

    prediction_export = predictions.copy()
    prediction_export["actual_label"] = actual_label
    prediction_export["image_path"] = str(image_path)
    prediction_rows.append(prediction_export)

    print(f"Image: {image_path}")
    print(f"Actual label: {actual_label}")
    display(predictions)
    display(action)

demo_predictions_df = pd.concat(prediction_rows, ignore_index=True)
demo_summary_df = pd.DataFrame(summary_rows)

demo_predictions_df.to_csv(CFG.RESULTS_DIR / CFG.DEMO_PREDICTIONS_FILE, index=False)
demo_summary_df.to_csv(CFG.RESULTS_DIR / CFG.DEMO_SUMMARY_FILE, index=False)

print(f"Saved predictions: {CFG.RESULTS_DIR / CFG.DEMO_PREDICTIONS_FILE}")
print(f"Saved decision summary: {CFG.RESULTS_DIR / CFG.DEMO_SUMMARY_FILE}")
print("Decision band counts")
display(demo_summary_df["decision_band"].value_counts().rename_axis("decision_band").reset_index(name="count"))

demo_summary_df


## 6. Run Insight

This notebook is the final bridge from model evaluation to user-facing behavior. The best product experience should not always force one class label. It should accept easy cases, show suggestions for ambiguous cases, and request confirmation or review for risky predictions.

The exported CSV files make the demo auditable outside the notebook and easier to summarize in project documentation.
